In [45]:
import sys
from pathlib import Path
from pyprojroot import here

sys.path.append(str(here()))

In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import load_data
from sklearn.model_selection import train_test_split
import torch
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import importlib
from src.config import cfg
from src.processing import (
    FinalTreeTransformer,
    MemoryOptimizer,
    new_num_columns,
    new_ohe_columns,
    new_ordinal_categories,
    new_ordinal_columns,
)

from src.utils import set_seed

In [47]:
import warnings

warnings.filterwarnings("ignore", message="Found unknown categories in columns")

In [48]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
gseed = cfg.general.seed
set_seed(gseed)

In [50]:
train_path = Path(cfg.paths.train)
test_path = Path(cfg.paths.test)

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
df_all_data = pd.concat(
    [df_train.drop(columns=["SalePrice"]), df_test], axis=0
).reset_index(drop=True)

In [51]:
X_train_orig, y_train_orig_raw, X_test, test_ids = load_data(cfg, False)

y_train_orig = np.log1p(y_train_orig_raw)

In [52]:
def get_nn_data(X_original, y_original, pipe, p=0.15):
    X = X_original.copy()
    y = y_original.copy()

    y_bins = pd.qcut(y, q=10, labels=False, duplicates="drop")

    X_train_val, X_hold, y_train_val, y_hold = train_test_split(
        X, y, test_size=0.15, random_state=gseed, stratify=y_bins
    )

    y_train_val_bins = pd.qcut(y_train_val, q=10, labels=False, duplicates="drop")

    X_train, X_opt, y_train, y_opt = train_test_split(
        X_train_val,
        y_train_val,
        test_size=p / (1 - p),
        random_state=gseed,
        stratify=y_train_val_bins,
    )

    # через пайплайн
    X_train = pipe.fit_transform(X_train, y_train)
    X_opt = pipe.transform(X_opt)
    X_hold = pipe.transform(X_hold)

    return X_train, y_train, X_opt, y_opt, X_hold, y_hold

In [ ]:
import src.nn as src_nn

importlib.reload(src_nn)

final_lin = src_nn.final_lin


X_train, y_train, X_opt, y_opt, X_hold, y_hold = get_nn_data(
    X_train_orig, y_train_orig, final_lin
)

model = src_nn.MLPRegressor(lr=3e-2, eval_set=(X_opt, y_opt))

print(X_train.shape)
print(X_opt.shape)
print(X_hold.shape)

(1020, 151)
(219, 151)
(219, 151)


### base data

In [64]:
model.fit(X_train, y_train)

d:\vs_projects\fp_houses\src\nn.py:112: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  X_val_t = torch.as_tensor(


nn:   0%|          | 0/500 [00:00<?, ?it/s]

[Early Stopping] Обучение остановлено на эпохе 55


,eval_set,"( LotFron...x 151 columns], ...)"
,hidden_size,128
,lr,0.003
,epochs,500
,patience,30
,scheduler_patience,10
,batch_size,32
,device,'cpu'
,seed,101
Name,Type,Value
model_,Sequential,"Sequential( ..., bias=True) )"


In [65]:
from sklearn.metrics import root_mean_squared_error

root_mean_squared_error(y_opt, model.predict(X_opt))

0.8333475197556254